In [ ]:
# Position-first fixed-candidate extension configuration.
%reload_ext autoreload
%autoreload 2
%cd ~/erc-src/cuneiform-ocr-sign-alignment-worktree
%env PATH=$HOME/.local/bin:$PATH

import os

import sign_alignment.pipeline as pp
import pipeline_3_candidate_test as pp3
from sign_alignment.data_source import LocalDataSource
from sign_alignment.detector import ModelConfig, TabletImageDetector
from sign_alignment.visualizer import ColorConfig

ANNOTATIONS_DIR = os.path.expanduser(
    "~/erc-work-data/data-of-cuneiform-ocr-data/filtered_annotations"
)
CONFIG_FILE = "configs/detr.py"
CHECKPOINT_FILE = os.path.expanduser(
    "~/erc-work-data/retrained_models/detr-173/epoch_1000.pth"
)
SCORE_THRESHOLD = 0.0
OUTPUT_DIR = "alignment_results_3_candidate_test"
SAMPLE_NAME = "NBC.4020"
SAMPLE_NAME = "YBC.7794"
SAMPLE_NAME = "HS.2020"
CROP_INDEX = 1


In [ ]:
# Initialize the shared base detector and context.
model_config = ModelConfig(
    config_file=CONFIG_FILE,
    checkpoint_file=CHECKPOINT_FILE,
    device="auto",
)
if "tablet_detector" not in globals() or getattr(tablet_detector, "model", None) is None:
    tablet_detector = TabletImageDetector(
        default_score_threshold=SCORE_THRESHOLD,
        model_config=model_config,
        keep_crops=True,
        is_crop_itself=False,
    )
else:
    print("Reusing existing tablet_detector instance.")

context = pp.CropContext(
    tablet_detector=tablet_detector,
    local_source=LocalDataSource(ANNOTATIONS_DIR),
    color_config=ColorConfig,
    output_dir=OUTPUT_DIR,
    img_idx=CROP_INDEX,
    task_type="candidate_test",
)
runner = pp.Runner(context, pp.VisOptions(info=True, display=True, save=True))


In [ ]:
# Run the base pipeline through coarse text alignment.
runner.choose_sample(name=SAMPLE_NAME)
runner.run([
    pp.Step("Load data", pp.load_data, pp.vis_loaded_data),
    pp.Step("Detect signs", pp.detect_signs, pp.vis_detections),
    pp.Step("Transform GT to crop", pp.transform_gt_to_crop, pp.vis_crop_ground_truth),
    pp.Step("Detection statistics", lambda _: None, pp.vis_detection_statistics),
    pp.Step("Create box sets", pp.create_box_sets, pp.vis_box_sets),
    pp.Step("Detect rows", pp.detect_rows, pp.vis_detected_rows_info),
    pp.Step("Match rows", pp.match_rows, pp.vis_row_matches),
    pp.Step("Visualize rows", lambda _: None, pp.vis_detection_rows),
    pp.Step("Match signs", pp.match_signs_in_rows, pp.vis_sign_matches),
    pp.Step("Align text rows", pp.align_text_rows, pp.vis_aligned_rows),
    pp.Step("Build sign match info", pp.build_sign_match_info, pp.vis_sign_match_info),
    pp.Step(
        "Result without optimization",
        pp.create_result_without_optimization,
        pp.vis_result_without_optimization,
    ),
    pp.Step("Unload detector", pp.unload_detector),
])


In [ ]:
# Run the extension without overwriting base alignment results.
candidate_config = pp3.CandidateAttractionConfig()
candidate_run = pp3.run_candidate_attraction(context, candidate_config)
runner.run([
    pp.Step("Candidate attraction", lambda _: None, pp3.vis_candidate_attraction),
    pp.Step(
        "Candidate alignment diagnostic",
        pp3.build_candidate_sign_match_info,
        pp3.vis_candidate_alignment_diagnostic,
    ),
    pp.Step("Candidate comparison", lambda _: None, pp3.vis_candidate_results_comparison),
])


In [ ]:
# Inspect all assignments, including position-only wrong-label matches.
records = pp3.candidate_attraction_records(context)
try:
    import pandas as pd

    candidate_table = pd.DataFrame(records)
    display(candidate_table[[
        "text_row_idx", "text_idx", "sign_name", "input_status",
        "output_status", "candidate_idx", "detector_labels",
        "class_support", "soft_probability", "null_probability",
        "included_in_result", "movement_px",
    ]])
except ImportError:
    candidate_table = records
    display(candidate_table[:20])

position_only = [
    row for row in records
    if row["output_status"] == "candidate" and row["class_support"] <= 0.0
]
print(f"Position-only/wrong-label candidate matches: {len(position_only)}")


In [ ]:
# Verify that extension output remains isolated from the base state.
state = context.state
print("base aligned boxes:", len(state.aligned_boxes))
print("candidate boxes:   ", len(candidate_run.boxes))
print("separate objects:  ", state.aligned_boxes is not candidate_run.boxes)
print("base final boxes:  ", state.final_boxes)
print("extension keys:    ", sorted(state.extras))
